# 🐦 BirdCLEF 2026 — Complete Fixed Multi-Resolution Submit Notebook

This is a full cleaned rewrite focused on eliminating the previous errors.

## Included

| Requirement | Status |
|---|---|
| Complete cell separation | ✅ |
| seaborn visualization | ✅ |
| CPU-only | ✅ |
| hidden test support | ✅ |
| row_id synchronization | ✅ |
| Proto 15-window + SED 12-window alignment | ✅ |
| Safe cache rebuild | ✅ |
| submit-ready `submission.csv` | ✅ |

## Design

- Proto branch uses **15 temporal points**: 4, 8, ..., 60 sec.
- Perch input stays **5 seconds** for model compatibility.
- SED branch stays **12 × 5 sec** because the SED ONNX model expects 5-sec mel shapes.
- Proto output is synchronized from **15 → 12** before final blending.

In [ ]:
# ============================================================
# CELL 1: Imports and CPU-only setup
# ============================================================

import os
import re
import gc
import json
import time
import random
import shutil
import warnings
import subprocess
import sys
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from scipy.ndimage import gaussian_filter1d

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.isotonic import IsotonicRegression

try:
    import onnxruntime as ort
    ORT_AVAILABLE = True
except Exception as e:
    ORT_AVAILABLE = False
    print("onnxruntime not available:", e)

try:
    import tensorflow as tf
    try:
        tf.config.set_visible_devices([], "GPU")
    except Exception:
        pass
    TF_AVAILABLE = True
except Exception as e:
    TF_AVAILABLE = False
    print("tensorflow not available:", e)

import torch
import torch.nn as nn
import torch.nn.functional as F

print("Imports complete")
print("ORT_AVAILABLE:", ORT_AVAILABLE)
print("TF_AVAILABLE :", TF_AVAILABLE)
print("Torch CUDA    :", torch.cuda.is_available())

In [ ]:
# ============================================================
# CELL 2: Seed, paths, and configuration
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    try:
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

seed_everything(42)

MODE = "submit"
assert MODE in {"submit", "train"}

BASE = Path("/kaggle/input/competitions/birdclef-2026")
WORK_DIR = Path("/kaggle/working")
CACHE_DIR = WORK_DIR / "cache_multires_fixed"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SR = 32_000
FILE_SEC = 60
FILE_SAMPLES = FILE_SEC * SR

PROTO_STEP_SEC = 4
PROTO_N_WINDOWS = 15
PROTO_ENDS = np.arange(PROTO_STEP_SEC, FILE_SEC + 1, PROTO_STEP_SEC)

PERCH_INPUT_SEC = 5
PERCH_INPUT_SAMPLES = PERCH_INPUT_SEC * SR

SED_WINDOW_SEC = 5
SED_N_WINDOWS = 12
SED_ENDS = np.arange(SED_WINDOW_SEC, FILE_SEC + 1, SED_WINDOW_SEC)
SED_WINDOW_SAMPLES = SED_WINDOW_SEC * SR

assert len(PROTO_ENDS) == PROTO_N_WINDOWS
assert len(SED_ENDS) == SED_N_WINDOWS

CFG = {
    "batch_files": 12,
    "force_rebuild_proto_cache": True,
    "dryrun_n_files": 20,
    "proto_epochs": 24 if MODE == "submit" else 60,
    "proto_patience": 6 if MODE == "submit" else 15,
    "run_oof": MODE == "train",
    "verbose": MODE == "train",
}

print("MODE:", MODE)
print("PROTO_ENDS:", PROTO_ENDS.tolist())
print("SED_ENDS  :", SED_ENDS.tolist())

In [ ]:
# ============================================================
# CELL 3: Seaborn visual setup
# ============================================================

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titleweight"] = "bold"

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(
    data=pd.DataFrame({
        "branch": ["Proto temporal grid", "SED / official grid"],
        "windows": [PROTO_N_WINDOWS, SED_N_WINDOWS],
    }),
    x="branch",
    y="windows",
    ax=ax,
)
ax.set_title("Multi-resolution design")
ax.set_xlabel("")
ax.set_ylabel("Windows per 60-sec file")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 4: Optional offline wheel installation helper
# ============================================================

INPUT_ROOT = Path("/kaggle/input")

def find_first(pattern):
    hits = list(INPUT_ROOT.rglob(pattern))
    return hits[0] if hits else None

if not ORT_AVAILABLE:
    whl = find_first("onnxruntime-*.whl")
    if whl is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(whl)], check=False)
        try:
            import onnxruntime as ort
            ORT_AVAILABLE = True
            print("ONNX Runtime installed from:", whl)
        except Exception as e:
            print("ONNX Runtime still unavailable:", e)

if not TF_AVAILABLE:
    tb = find_first("tensorboard-*.whl")
    tf_whl = find_first("tensorflow-*.whl")
    if tb is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(tb)], check=False)
    if tf_whl is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(tf_whl)], check=False)
    try:
        import tensorflow as tf
        try:
            tf.config.set_visible_devices([], "GPU")
        except Exception:
            pass
        TF_AVAILABLE = True
        print("TensorFlow installed")
    except Exception as e:
        print("TensorFlow still unavailable:", e)

In [ ]:
# ============================================================
# CELL 5: Data loading and official 12-window labels
# ============================================================

taxonomy = pd.read_csv(BASE / "taxonomy.csv")
sample_sub = pd.read_csv(BASE / "sample_submission.csv")
soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)
label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}

FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_fname(name):
    m = FNAME_RE.match(str(name))
    if not m:
        return {"site": "unknown", "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}

def union_labels(series):
    out = set()
    for x in series:
        if pd.notna(x):
            for t in str(x).split(";"):
                t = t.strip()
                if t:
                    out.add(t)
    return sorted(out)

sc = (
    soundscape_labels
    .groupby(["filename", "start", "end"])["primary_label"]
    .apply(union_labels)
    .reset_index(name="label_list")
)

sc["end_sec"] = pd.to_timedelta(sc["end"]).dt.total_seconds().astype(int)
sc["row_id"] = sc["filename"].str.replace(".ogg", "", regex=False) + "_" + sc["end_sec"].astype(str)

name_meta = sc["filename"].apply(parse_fname).apply(pd.Series)
sc = pd.concat([sc, name_meta], axis=1)

Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
for i, labels in enumerate(sc["label_list"]):
    for lbl in labels:
        if lbl in label_to_idx:
            Y_SC[i, label_to_idx[lbl]] = 1

windows_per_file = sc.groupby("filename").size()
full_files = sorted(windows_per_file[windows_per_file == SED_N_WINDOWS].index.tolist())

full_rows_12 = (
    sc[sc["filename"].isin(full_files)]
    .sort_values(["filename", "end_sec"])
    .reset_index(drop=False)
)

Y_12 = Y_SC[full_rows_12["index"].to_numpy()]
file_order_12 = full_rows_12.drop_duplicates("filename")["filename"].tolist()

print("N_CLASSES:", N_CLASSES)
print("Full 12-window train files:", len(full_files))
print("Y_12:", Y_12.shape)

In [ ]:
# ============================================================
# CELL 6: Project official 12-window labels to Proto 15-window grid
# ============================================================

def project_12_to_15_labels(Y12):
    n_files = len(Y12) // SED_N_WINDOWS
    Y12_f = Y12.reshape(n_files, SED_N_WINDOWS, -1)

    nearest_idx = np.array([
        int(np.argmin(np.abs(SED_ENDS - t)))
        for t in PROTO_ENDS
    ], dtype=np.int64)

    Y15_f = Y12_f[:, nearest_idx, :]
    return Y15_f.reshape(n_files * PROTO_N_WINDOWS, -1)

Y_15 = project_12_to_15_labels(Y_12)

print("Y_15:", Y_15.shape)

label_counts = pd.Series(Y_12.sum(axis=0), index=PRIMARY_LABELS).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(x=label_counts.head(30).values, y=label_counts.head(30).index, ax=ax)
ax.set_title("Top 30 target labels in official 12-window labels")
ax.set_xlabel("Positive windows")
ax.set_ylabel("Primary label")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 7: Perch model loading and species mapping
# ============================================================

MODEL_DIR_CANDIDATES = [
    Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1"),
    Path("/kaggle/input/models/google/bird-vocalization-classifier/TensorFlow2/perch_v2_cpu/1"),
]

MODEL_DIR = next((p for p in MODEL_DIR_CANDIDATES if p.exists()), None)
if MODEL_DIR is None:
    raise FileNotFoundError("Google Perch model directory not found under /kaggle/input/models")

ONNX_PERCH_CANDIDATES = list(Path("/kaggle/input").rglob("perch_v2.onnx"))
ONNX_PERCH_PATH = ONNX_PERCH_CANDIDATES[0] if ONNX_PERCH_CANDIDATES else None

USE_ONNX = ORT_AVAILABLE and ONNX_PERCH_PATH is not None

if USE_ONNX:
    so = ort.SessionOptions()
    so.intra_op_num_threads = 4
    so.inter_op_num_threads = 1
    ONNX_SESSION = ort.InferenceSession(str(ONNX_PERCH_PATH), sess_options=so, providers=["CPUExecutionProvider"])
    ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
    ONNX_OUT_MAP = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
    print("Using ONNX Perch:", ONNX_PERCH_PATH)
else:
    if not TF_AVAILABLE:
        raise RuntimeError("Neither ONNX Perch nor TensorFlow Perch is available.")
    birdclassifier = tf.saved_model.load(str(MODEL_DIR))
    infer_fn = birdclassifier.signatures["serving_default"]
    print("Using TensorFlow Perch")

labels_path = MODEL_DIR / "assets" / "labels.csv"
bc_labels = (
    pd.read_csv(labels_path)
    .reset_index()
    .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
)
NO_LABEL = len(bc_labels)

mapping = taxonomy.merge(
    bc_labels[["bc_index", "scientific_name"]],
    on="scientific_name",
    how="left",
)
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL).astype(int)
lbl2bc = mapping.set_index("primary_label")["bc_index"]

BC_INDICES = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK = BC_INDICES != NO_LABEL
MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)

CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()

proxy_map = {}
UNMAPPED_POS = np.where(~MAPPED_MASK)[0].astype(np.int32)

for idx in UNMAPPED_POS:
    target = PRIMARY_LABELS[idx]
    row = taxonomy[taxonomy["primary_label"] == target]
    if len(row) == 0:
        continue
    sci = str(row.iloc[0]["scientific_name"])
    genus = sci.split()[0]
    hits = bc_labels[
        bc_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)
    ]
    if len(hits) > 0:
        proxy_map[idx] = hits["bc_index"].astype(int).tolist()

print(f"Exact Perch mapped: {MAPPED_MASK.sum()} / {N_CLASSES}")
print(f"Genus proxy mapped: {len(proxy_map)}")

coverage_df = pd.DataFrame({
    "mapping": ["Exact", "Proxy", "No direct"],
    "count": [int(MAPPED_MASK.sum()), int(len(proxy_map)), int(N_CLASSES - MAPPED_MASK.sum() - len(proxy_map))],
})
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=coverage_df, x="mapping", y="count", ax=ax)
ax.set_title("Perch mapping coverage")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 8: Audio helpers and Proto 15-window Perch inference
# ============================================================

def read_audio_60s(path):
    y, sr0 = sf.read(str(path), dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr0 != SR:
        y = librosa.resample(y, orig_sr=sr0, target_sr=SR)
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    else:
        y = y[:FILE_SAMPLES]
    return y.astype(np.float32)

def extract_centered_excerpt(y, end_sec, length_sec=PERCH_INPUT_SEC):
    center = int(end_sec * SR)
    half = int(length_sec * SR // 2)
    start = center - half
    end = start + int(length_sec * SR)

    if start < 0:
        start = 0
        end = start + int(length_sec * SR)
    if end > FILE_SAMPLES:
        end = FILE_SAMPLES
        start = end - int(length_sec * SR)

    chunk = y[start:end]
    if len(chunk) < int(length_sec * SR):
        chunk = np.pad(chunk, (0, int(length_sec * SR) - len(chunk)))
    return chunk.astype(np.float32)

def run_perch_proto15(paths, batch_files=12, verbose=True):
    paths = [Path(p) for p in paths]
    n_rows = len(paths) * PROTO_N_WINDOWS

    row_ids = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites = np.empty(n_rows, dtype=object)
    hours = np.zeros(n_rows, dtype=np.int16)

    scores = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embs = np.zeros((n_rows, 1536), dtype=np.float32)

    global_wr = 0
    iterator = range(0, len(paths), batch_files)
    if verbose:
        iterator = tqdm(iterator, desc="Perch Proto15")

    for start in iterator:
        batch_paths = paths[start:start + batch_files]
        batch_chunks = []
        local_meta = []

        for p in batch_paths:
            y = read_audio_60s(p)
            meta = parse_fname(p.name)
            stem = p.stem

            for end_sec in PROTO_ENDS:
                batch_chunks.append(extract_centered_excerpt(y, int(end_sec)))
                local_meta.append((f"{stem}_{int(end_sec)}", p.name, meta["site"], meta["hour_utc"]))

        x = np.stack(batch_chunks).astype(np.float32)

        if USE_ONNX:
            outs = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: x})
            logits = outs[ONNX_OUT_MAP["label"]].astype(np.float32)
            emb = outs[ONNX_OUT_MAP["embedding"]].astype(np.float32)
        else:
            out = infer_fn(inputs=tf.convert_to_tensor(x))
            logits = out["label"].numpy().astype(np.float32)
            emb = out["embedding"].numpy().astype(np.float32)

        br = global_wr
        er = global_wr + len(batch_chunks)

        for i, (rid, fname, site, hour) in enumerate(local_meta, start=br):
            row_ids[i] = rid
            filenames[i] = fname
            sites[i] = site
            hours[i] = hour

        scores[br:er, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
        embs[br:er] = emb

        for pos_idx, bc_idxs in proxy_map.items():
            bc_arr = np.array(bc_idxs, dtype=np.int32)
            scores[br:er, pos_idx] = logits[:, bc_arr].max(axis=1)

        global_wr = er

        del x, logits, emb, batch_chunks, local_meta
        gc.collect()

    meta_df = pd.DataFrame({
        "row_id": row_ids,
        "filename": filenames,
        "site": sites,
        "hour_utc": hours,
    })

    assert len(meta_df) == len(scores) == len(embs)
    assert len(meta_df) % PROTO_N_WINDOWS == 0

    return meta_df, scores, embs

print("Proto15 Perch inference ready")

In [ ]:
# ============================================================
# CELL 9: Safe Proto15 cache rebuild
# ============================================================

CACHE_META = CACHE_DIR / "proto15_meta.parquet"
CACHE_NPZ = CACHE_DIR / "proto15_arrays.npz"

def build_or_load_proto_cache():
    if CFG["force_rebuild_proto_cache"]:
        for p in [CACHE_META, CACHE_NPZ]:
            if p.exists():
                p.unlink()

    if CACHE_META.exists() and CACHE_NPZ.exists():
        print("Loading Proto15 cache:", CACHE_DIR)
        meta = pd.read_parquet(CACHE_META)
        arr = np.load(CACHE_NPZ)
        scores = arr["scores"].astype(np.float32)
        embs = arr["embs"].astype(np.float32)
    else:
        print("Building Proto15 cache from train_soundscapes")
        train_paths = [BASE / "train_soundscapes" / fn for fn in full_files]
        train_paths = [p for p in train_paths if p.exists()]
        print("Train paths:", len(train_paths))

        meta, scores, embs = run_perch_proto15(
            train_paths,
            batch_files=CFG["batch_files"],
            verbose=True
        )

        meta.to_parquet(CACHE_META)
        np.savez(
            CACHE_NPZ,
            scores=scores.astype(np.float32),
            embs=embs.astype(np.float32),
            primary_labels=np.array(PRIMARY_LABELS),
        )
        print("Proto15 cache saved")

    assert len(meta) % PROTO_N_WINDOWS == 0
    assert len(scores) == len(meta)
    assert len(embs) == len(meta)

    return meta, scores, embs

meta_tr15, sc_tr15, emb_tr15 = build_or_load_proto_cache()
Y_FULL15 = Y_15

assert len(Y_FULL15) == len(meta_tr15), f"Y_FULL15 {len(Y_FULL15)} != meta_tr15 {len(meta_tr15)}"

shape_df = pd.DataFrame({
    "object": ["meta_tr15", "sc_tr15", "emb_tr15", "Y_FULL15"],
    "rows": [len(meta_tr15), len(sc_tr15), len(emb_tr15), len(Y_FULL15)]
})
display(shape_df)

fig, ax = plt.subplots(figsize=(12, 5))
sample_scores = pd.Series(sc_tr15.ravel()).sample(min(200_000, sc_tr15.size), random_state=42)
sns.histplot(sample_scores, bins=80, kde=True, ax=ax)
ax.set_title("Proto15 Perch score distribution")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 10: Prior and post-processing helpers
# ============================================================

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

def build_prior_tables(meta_df, Y):
    df = meta_df.reset_index(drop=True)
    global_p = Y.mean(axis=0).astype(np.float32)

    site_keys = sorted(df["site"].dropna().astype(str).unique())
    site_to_i = {s: i for i, s in enumerate(site_keys)}
    site_p = np.zeros((len(site_keys), Y.shape[1]), dtype=np.float32)
    site_n = np.zeros(len(site_keys), dtype=np.float32)

    for s in site_keys:
        idx = site_to_i[s]
        mask = df["site"].astype(str).values == s
        site_n[idx] = mask.sum()
        site_p[idx] = Y[mask].mean(axis=0)

    hour_keys = sorted(df["hour_utc"].dropna().astype(int).unique())
    hour_to_i = {h: i for i, h in enumerate(hour_keys)}
    hour_p = np.zeros((len(hour_keys), Y.shape[1]), dtype=np.float32)
    hour_n = np.zeros(len(hour_keys), dtype=np.float32)

    for h in hour_keys:
        idx = hour_to_i[h]
        mask = df["hour_utc"].astype(int).values == h
        hour_n[idx] = mask.sum()
        hour_p[idx] = Y[mask].mean(axis=0)

    return {
        "global_p": global_p,
        "site_to_i": site_to_i,
        "site_p": site_p,
        "site_n": site_n,
        "hour_to_i": hour_to_i,
        "hour_p": hour_p,
        "hour_n": hour_n,
    }

def apply_prior(scores, sites, hours, tables, lambda_prior=0.35):
    eps = 1e-4
    n = len(scores)
    p = np.tile(tables["global_p"], (n, 1))

    for i, h in enumerate(hours):
        h = int(h)
        if h in tables["hour_to_i"]:
            j = tables["hour_to_i"][h]
            w = tables["hour_n"][j] / (tables["hour_n"][j] + 8.0)
            p[i] = w * tables["hour_p"][j] + (1 - w) * p[i]

    for i, s in enumerate(sites):
        s = str(s)
        if s in tables["site_to_i"]:
            j = tables["site_to_i"][s]
            w = tables["site_n"][j] / (tables["site_n"][j] + 8.0)
            p[i] = w * tables["site_p"][j] + (1 - w) * p[i]

    p = np.clip(p, eps, 1 - eps)
    logit_prior = np.log(p) - np.log1p(-p)
    return (scores + lambda_prior * logit_prior).astype(np.float32)

def file_confidence_scale(probs, n_windows, top_k=2, power=0.4):
    N, C = probs.shape
    assert N % n_windows == 0
    x = probs.reshape(-1, n_windows, C)
    top = np.sort(x, axis=1)[:, -top_k:, :].mean(axis=1, keepdims=True)
    return (x * np.power(top, power)).reshape(N, C)

def rank_aware_scaling(probs, n_windows, power=0.35):
    N, C = probs.shape
    assert N % n_windows == 0
    x = probs.reshape(-1, n_windows, C)
    file_max = x.max(axis=1, keepdims=True)
    return (x * np.power(file_max, power)).reshape(N, C)

def adaptive_delta_smooth(probs, n_windows, base_alpha=0.20):
    N, C = probs.shape
    assert N % n_windows == 0
    x = probs.reshape(-1, n_windows, C)
    out = x.copy()
    for t in range(n_windows):
        conf = x[:, t, :].max(axis=-1, keepdims=True)
        alpha = base_alpha * (1.0 - conf)
        if t == 0:
            neigh = (x[:, t, :] + x[:, t + 1, :]) / 2.0
        elif t == n_windows - 1:
            neigh = (x[:, t - 1, :] + x[:, t, :]) / 2.0
        else:
            neigh = (x[:, t - 1, :] + x[:, t + 1, :]) / 2.0
        out[:, t, :] = (1 - alpha) * x[:, t, :] + alpha * neigh
    return out.reshape(N, C)

print("Prior and post-processing helpers ready")

In [ ]:
# ============================================================
# CELL 11: Light ProtoSSM model
# ============================================================

class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d = nn.Conv1d(d_model, d_model, d_conv, padding=d_conv - 1, groups=d_model)
        self.dt_proj = nn.Linear(d_model, d_model)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.d_state = d_state

    def forward(self, x):
        B, T, D = x.shape
        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)
        x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :T].transpose(1, 2)
        x_conv = F.silu(x_conv)

        dt = F.softplus(self.dt_proj(x_conv))
        A = -torch.exp(self.A_log)
        Bp = self.B_proj(x_conv)
        Cp = self.C_proj(x_conv)

        h = torch.zeros(B, D, self.d_state, device=x.device)
        ys = []
        for t in range(T):
            dA = torch.exp(A[None] * dt[:, t, :, None])
            dB = dt[:, t, :, None] * Bp[:, t, None, :]
            h = h * dA + x[:, t, :, None] * dB
            ys.append((h * Cp[:, t, None, :]).sum(-1))
        y = torch.stack(ys, dim=1)
        return self.out_proj(y * F.silu(z)) + x * self.D[None, None, :]

class LightProtoSSM(nn.Module):
    def __init__(self, d_input=1536, d_model=128, d_state=16, n_classes=234, n_windows=15, dropout=0.15, n_sites=20, meta_dim=16):
        super().__init__()
        self.n_windows = n_windows
        self.n_classes = n_classes

        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)

        self.ssm_fwd = SelectiveSSM(d_model, d_state)
        self.ssm_bwd = SelectiveSSM(d_model, d_state)
        self.merge = nn.Linear(2 * d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads=2, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

        self.prototypes = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp = nn.Parameter(torch.tensor(5.0))
        self.class_bias = nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))

    def init_prototypes(self, emb_flat, labels_flat):
        with torch.no_grad():
            h = self.input_proj(emb_flat)
            for c in range(self.n_classes):
                mask = labels_flat[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)

    def forward(self, emb, perch_logits=None, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(emb) + self.pos_enc[:, :T, :]

        if site_ids is not None and hours is not None:
            meta = self.meta_proj(torch.cat([self.site_emb(site_ids), self.hour_emb(hours)], dim=-1))
            h = h + meta[:, None, :]

        res = h
        hf = self.ssm_fwd(h)
        hb = self.ssm_bwd(h.flip(1)).flip(1)
        h = self.drop(self.merge(torch.cat([hf, hb], dim=-1)))
        h = self.norm(h + res)

        attn_out, _ = self.attn(h, h, h)
        h = self.attn_norm(h + attn_out)

        hn = F.normalize(h, dim=-1)
        pn = F.normalize(self.prototypes, dim=-1)
        sim = torch.matmul(hn, pn.T) * F.softplus(self.proto_temp) + self.class_bias[None, None, :]

        if perch_logits is not None:
            alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
            return alpha * sim + (1.0 - alpha) * perch_logits
        return sim

def get_file_level_site_hour(meta_df):
    fnames = meta_df.drop_duplicates("filename")["filename"].tolist()
    sites = []
    hours = []
    for fn in fnames:
        rows = meta_df[meta_df["filename"] == fn]
        sites.append(rows["site"].iloc[0])
        hours.append(int(rows["hour_utc"].iloc[0]) % 24)
    return fnames, np.array(sites), np.array(hours)

def train_light_proto_ssm(emb_full, scores_full, Y_full, meta_full, n_epochs=24, patience=6, lr=1e-3, n_sites=20):
    assert len(emb_full) % PROTO_N_WINDOWS == 0
    n_files = len(emb_full) // PROTO_N_WINDOWS

    emb_f = emb_full.reshape(n_files, PROTO_N_WINDOWS, -1)
    log_f = scores_full.reshape(n_files, PROTO_N_WINDOWS, -1)
    lab_f = Y_full.reshape(n_files, PROTO_N_WINDOWS, -1).astype(np.float32)

    fnames, site_names, hour_vals = get_file_level_site_hour(meta_full)
    site_keys = sorted(pd.Series(site_names).astype(str).unique())
    site2i = {s: i + 1 for i, s in enumerate(site_keys)}
    site_ids = np.array([min(site2i.get(str(s), 0), n_sites - 1) for s in site_names], dtype=np.int64)
    hour_ids = hour_vals.astype(np.int64) % 24

    model = LightProtoSSM(n_classes=N_CLASSES, n_windows=PROTO_N_WINDOWS, n_sites=n_sites)
    model.init_prototypes(torch.tensor(emb_full, dtype=torch.float32), torch.tensor(Y_full, dtype=torch.float32))

    emb_t = torch.tensor(emb_f, dtype=torch.float32)
    log_t = torch.tensor(log_f, dtype=torch.float32)
    lab_t = torch.tensor(lab_f, dtype=torch.float32)
    site_t = torch.tensor(site_ids, dtype=torch.long)
    hour_t = torch.tensor(hour_ids, dtype=torch.long)

    pos_cnt = lab_t.sum(dim=(0, 1))
    total = lab_t.shape[0] * lab_t.shape[1]
    pos_weight = ((total - pos_cnt) / (pos_cnt + 1.0)).clamp(max=25.0)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)

    best_loss = float("inf")
    best_state = None
    wait = 0

    for ep in range(n_epochs):
        model.train()
        out = model(emb_t, log_t, site_ids=site_t, hours=hour_t)
        loss = F.binary_cross_entropy_with_logits(out, lab_t, pos_weight=pos_weight[None, None, :])
        loss = loss + 0.10 * F.mse_loss(out, log_t)

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    return model, site2i

def predict_proto(model, emb_flat, scores_flat, meta_df, site2i, n_sites=20):
    assert len(emb_flat) % PROTO_N_WINDOWS == 0
    n_files = len(emb_flat) // PROTO_N_WINDOWS

    emb_f = emb_flat.reshape(n_files, PROTO_N_WINDOWS, -1)
    sc_f = scores_flat.reshape(n_files, PROTO_N_WINDOWS, -1)

    fnames, site_names, hour_vals = get_file_level_site_hour(meta_df)
    site_ids = np.array([min(site2i.get(str(s), 0), n_sites - 1) for s in site_names], dtype=np.int64)
    hour_ids = hour_vals.astype(np.int64) % 24

    with torch.no_grad():
        out = model(
            torch.tensor(emb_f, dtype=torch.float32),
            torch.tensor(sc_f, dtype=torch.float32),
            site_ids=torch.tensor(site_ids, dtype=torch.long),
            hours=torch.tensor(hour_ids, dtype=torch.long),
        ).numpy()

    return out.reshape(-1, N_CLASSES).astype(np.float32)

print("LightProtoSSM ready")

In [ ]:
# ============================================================
# CELL 12: Train Proto branch and run test/dry-run inference
# ============================================================

test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
IS_DRY_RUN = len(test_paths) == 0

if IS_DRY_RUN:
    n = CFG["dryrun_n_files"] or 20
    print(f"No hidden test found. Dry-run using {n} train files.")
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:n]
else:
    print("Hidden test files:", len(test_paths))

t0 = time.time()
proto_model, site2i_tr = train_light_proto_ssm(
    emb_tr15,
    sc_tr15,
    Y_FULL15,
    meta_tr15,
    n_epochs=CFG["proto_epochs"],
    patience=CFG["proto_patience"],
    lr=1e-3,
)
print(f"ProtoSSM trained in {time.time() - t0:.1f}s")

meta_te15, sc_te15, emb_te15 = run_perch_proto15(
    test_paths,
    batch_files=CFG["batch_files"],
    verbose=True
)
print("Test Proto15 shapes:", meta_te15.shape, sc_te15.shape, emb_te15.shape)

prior_tables = build_prior_tables(meta_tr15, Y_FULL15)
sc_te_prior = apply_prior(
    sc_te15,
    sites=meta_te15["site"].to_numpy(),
    hours=meta_te15["hour_utc"].to_numpy(),
    tables=prior_tables,
    lambda_prior=0.35,
)

proto_logits15 = predict_proto(proto_model, emb_te15, sc_te_prior, meta_te15, site2i_tr)

probs_proto15 = sigmoid(proto_logits15)
probs_proto15 = file_confidence_scale(probs_proto15, n_windows=PROTO_N_WINDOWS, top_k=2, power=0.35)
probs_proto15 = rank_aware_scaling(probs_proto15, n_windows=PROTO_N_WINDOWS, power=0.30)
probs_proto15 = adaptive_delta_smooth(probs_proto15, n_windows=PROTO_N_WINDOWS, base_alpha=0.20)
probs_proto15 = np.clip(probs_proto15, 0.0, 1.0)

print("probs_proto15:", probs_proto15.shape)

In [ ]:
# ============================================================
# CELL 13: Temporal synchronization 15 -> 12
# ============================================================

def align_15_to_12(probs15):
    assert len(probs15) % PROTO_N_WINDOWS == 0
    n_files = len(probs15) // PROTO_N_WINDOWS
    C = probs15.shape[1]
    x = probs15.reshape(n_files, PROTO_N_WINDOWS, C)

    out = np.zeros((n_files, SED_N_WINDOWS, C), dtype=np.float32)

    for f in range(n_files):
        for c in range(C):
            out[f, :, c] = np.interp(SED_ENDS, PROTO_ENDS, x[f, :, c])

    return out.reshape(n_files * SED_N_WINDOWS, C)

def make_official_row_ids_from_paths(paths):
    rows = []
    for p in paths:
        stem = Path(p).stem
        for t in SED_ENDS:
            rows.append(f"{stem}_{int(t)}")
    return rows

probs_proto12 = align_15_to_12(probs_proto15)
proto12_rows = make_official_row_ids_from_paths(test_paths)

submission_proto = pd.DataFrame(probs_proto12.astype(np.float32), columns=PRIMARY_LABELS)
submission_proto.insert(0, "row_id", proto12_rows)
submission_proto.to_csv("submission_protossm.csv", index=False)

print("submission_protossm.csv saved:", submission_proto.shape)

fig, ax = plt.subplots(figsize=(12, 5))
sampled = pd.Series(probs_proto12.ravel()).sample(min(200_000, probs_proto12.size), random_state=42)
sns.histplot(sampled, bins=80, kde=True, ax=ax)
ax.set_title("Proto branch probability distribution after 15→12 alignment")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 14: SED branch fixed at 5 sec / 12 windows
# ============================================================

N_MELS_SED = 256
N_FFT_SED = 2048
HOP_SED = 512
FMIN_SED = 20
FMAX_SED = 16000
TOP_DB_SED = 80

def find_sed_dir():
    hits = sorted(Path("/kaggle/input").rglob("sed_fold0.onnx"))
    if not hits:
        return None
    return hits[0].parent

def make_sed_session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 4
    so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), sess_options=so, providers=["CPUExecutionProvider"])

def file_to_sed_chunks(path):
    y = read_audio_60s(path)
    chunks = y.reshape(SED_N_WINDOWS, SED_WINDOW_SAMPLES)
    ends = SED_ENDS.copy()
    return chunks, ends

def audio_to_mel(chunks):
    mels = []
    for x in chunks:
        s = librosa.feature.melspectrogram(
            y=x,
            sr=SR,
            n_fft=N_FFT_SED,
            hop_length=HOP_SED,
            n_mels=N_MELS_SED,
            fmin=FMIN_SED,
            fmax=FMAX_SED,
            power=2.0,
        )
        s = librosa.power_to_db(s, top_db=TOP_DB_SED)
        s = (s - s.mean()) / (s.std() + 1e-6)
        mels.append(s)
    return np.stack(mels)[:, None].astype(np.float32)

def sigmoid_sed(x):
    return (1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))).astype(np.float32)

sed_dir = find_sed_dir()
sed_available = (sed_dir is not None) and ORT_AVAILABLE

if sed_available:
    sed_fold_paths = sorted(
        sed_dir.glob("sed_fold*.onnx"),
        key=lambda p: int(re.search(r"sed_fold(\d+)", p.name).group(1))
    )
    print("SED folds:", [p.name for p in sed_fold_paths])
    sed_sessions = [make_sed_session(p) for p in sed_fold_paths]

    sed_rows = []
    sed_preds = []

    for i, path in enumerate(tqdm(test_paths, desc="SED inference"), 1):
        chunks, ends = file_to_sed_chunks(path)
        mel = audio_to_mel(chunks)

        if i == 1:
            print("First mel shape:", mel.shape)

        p_sum = np.zeros((len(chunks), N_CLASSES), dtype=np.float32)

        for sess in sed_sessions:
            outs = sess.run(None, {sess.get_inputs()[0].name: mel})
            clip_logits = outs[0]
            frame_max = outs[1].max(axis=1)
            p_sum += 0.5 * sigmoid_sed(clip_logits) + 0.5 * sigmoid_sed(frame_max)

        p_mean = p_sum / max(len(sed_sessions), 1)

        if len(p_mean) > 1:
            p_mean = gaussian_filter1d(p_mean, sigma=0.65, axis=0, mode="nearest").astype(np.float32)

        stem = Path(path).stem
        sed_rows.extend([f"{stem}_{int(t)}" for t in ends])
        sed_preds.append(p_mean)

    sed_preds_arr = np.concatenate(sed_preds, axis=0)
    submission_sed = pd.DataFrame(np.clip(sed_preds_arr, 0, 1), columns=PRIMARY_LABELS)
    submission_sed.insert(0, "row_id", sed_rows)
    submission_sed.to_csv("submission_sed.csv", index=False)
    print("submission_sed.csv saved:", submission_sed.shape)

else:
    print("SED ONNX not found or ONNX unavailable. Using Proto-only fallback.")
    submission_sed = submission_proto.copy()
    submission_sed.to_csv("submission_sed.csv", index=False)

In [ ]:
# ============================================================
# CELL 15: Safe rank blend and final submission
# ============================================================

EPS = 1e-5

df_proto = pd.read_csv("submission_protossm.csv")
df_sed = pd.read_csv("submission_sed.csv")

cols = [c for c in df_proto.columns if c != "row_id"]

sed_index = df_sed.set_index("row_id")
missing = [r for r in df_proto["row_id"].astype(str).tolist() if r not in sed_index.index]

if missing:
    print("SED row_id mismatch. Falling back to Proto-only for missing rows.")
    df_sed = df_proto.copy()
else:
    df_sed = sed_index.loc[df_proto["row_id"].astype(str)].reset_index()

p_proto = np.clip(df_proto[cols].to_numpy(np.float32), EPS, 1 - EPS)
p_sed = np.clip(df_sed[cols].to_numpy(np.float32), EPS, 1 - EPS)

rank_proto = pd.DataFrame(p_proto).rank(axis=0, pct=True).to_numpy(np.float32)
rank_sed = pd.DataFrame(p_sed).rank(axis=0, pct=True).to_numpy(np.float32)

pred = 0.60 * rank_proto + 0.40 * rank_sed

row_ids = df_proto["row_id"].astype(str).to_numpy()
file_ids = np.array(["_".join(r.split("_")[:-1]) for r in row_ids])

offs = np.arange(-3, 4, dtype=np.float32)
kernel = (1.0 + (offs / 1.20) ** 2 / 2.0) ** (-1.5)
kernel = (kernel / kernel.sum()).astype(np.float32)

ctx = p_proto.copy()
for fid in pd.unique(file_ids):
    m = file_ids == fid
    x = p_proto[m]
    if len(x) > 1:
        xp = np.pad(x, ((3, 3), (0, 0)), mode="edge")
        ctx[m] = sum(kernel[i] * xp[i:i + len(x)] for i in range(7))

ctx_rank = pd.DataFrame(ctx).rank(axis=0, pct=True).to_numpy(np.float32)
fake_only = (p_proto > 0.50) & (p_sed < 0.05)
proto_cont = (ctx_rank > 0.88) & (rank_proto > 0.75) & (p_sed < 0.12) & (~fake_only)

pred = np.where(fake_only, 0.92 * pred + 0.08 * rank_proto, pred)
pred = np.where(proto_cont, 0.85 * pred + 0.15 * np.maximum(rank_proto, ctx_rank), pred)

sed_only = (rank_sed > 0.95) & (rank_proto < 0.80) & (~fake_only) & (~proto_cont)
pred = np.where(sed_only, 0.88 * pred + 0.12 * rank_sed, pred)

sub = df_proto.copy()
sub[cols] = np.clip(pred, 0, 1).astype(np.float32)

MIRROR_PAIRS = (
    ("47158son15", "47158son16"),
    ("47158son09", "47158son12"),
    ("47158son02", "47158son14"),
    ("47158son13", "47158son21", "47158son22", "47158son23"),
)
col_to_idx = {c: i for i, c in enumerate(cols)}

mirror_count = 0
for group in MIRROR_PAIRS:
    valid = [col_to_idx[x] for x in group if x in col_to_idx]
    if len(valid) >= 2:
        group_max = sub[cols].iloc[:, valid].max(axis=1).to_numpy(np.float32)
        for idx in valid:
            sub.iloc[:, idx + 1] = group_max
        mirror_count += len(valid)

print("Sonotype mirroring columns:", mirror_count)

try:
    tax_df = taxonomy.set_index("primary_label")
    rare_classes = {"Amphibia", "Mammalia", "Reptilia"}
    rare_count = 0
    for ci, species in enumerate(cols):
        if species in tax_df.index and tax_df.loc[species, "class_name"] in rare_classes:
            vals = sub.iloc[:, ci + 1].to_numpy(np.float32)
            thr = np.percentile(vals, 75)
            sub.iloc[:, ci + 1] = np.where(vals < thr, vals * 0.90, vals)
            rare_count += 1
    print("Rare suppression applied:", rare_count)
except Exception as e:
    print("Rare suppression skipped:", e)

if IS_DRY_RUN:
    print("Dry-run detected. Aligning final output to sample_submission.csv.")
    template = sub[cols].mean(axis=0).astype(np.float32)
    final_sub = sample_sub.copy()
    for c in cols:
        final_sub[c] = template[c]
    sub = final_sub

assert list(sub.columns) == list(sample_sub.columns), "Column mismatch with sample_submission.csv"
sub[cols] = sub[cols].clip(0, 1).astype(np.float32)

sub.to_csv("submission.csv", index=False)

print("submission.csv saved:", sub.shape)
print("Value range:", float(sub[cols].min().min()), float(sub[cols].max().max()))

In [ ]:
# ============================================================
# CELL 16: Final seaborn diagnostics
# ============================================================

final_sub = pd.read_csv("submission.csv")
pred_cols = [c for c in final_sub.columns if c != "row_id"]
vals = final_sub[pred_cols].to_numpy(np.float32)

print("submission shape:", final_sub.shape)
print("min / max / mean:", vals.min(), vals.max(), vals.mean())

fig, ax = plt.subplots(figsize=(12, 5))
sample_vals = pd.Series(vals.ravel()).sample(min(200_000, vals.size), random_state=42)
sns.histplot(sample_vals, bins=80, kde=True, ax=ax)
ax.set_title("Final submission prediction distribution")
ax.set_xlabel("Prediction")
plt.tight_layout()
plt.show()

mean_pred = final_sub[pred_cols].mean().sort_values(ascending=False).head(25)
fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(x=mean_pred.values, y=mean_pred.index, ax=ax)
ax.set_title("Top 25 classes by mean final prediction")
ax.set_xlabel("Mean prediction")
ax.set_ylabel("Primary label")
plt.tight_layout()
plt.show()

# ✅ Final checklist

| Check | Expected |
|---|---|
| `submission.csv` exists | ✅ |
| Columns match sample submission | ✅ |
| Proto branch uses 15 time points | ✅ |
| SED branch remains fixed to 12 official rows | ✅ |
| Proto 15→12 temporal sync | ✅ |
| hidden test support | ✅ |
| dry-run fallback | ✅ |
| CPU-only | ✅ |

This notebook intentionally does **not** reuse the old external 12-window Perch cache for the Proto15 branch.  
That is what prevents the old `12 vs 15 reshape` and `row_id mismatch` errors.